### ***Autograd***
- ***Autograd is a reverse-mode automatic differentiation framework that dynamically constructs a directed acyclic graph (DAG) during the forward pass to track tensor operations, enabling automatic computation of vector-Jacobian products (gradients) via backpropagation when calling .backward().***

In [1]:
# manually caluclate differentiation
def dy_dx(x):
  return 2*x

dy_dx(3)

6

In [2]:
import torch
x = torch.tensor(3.0,requires_grad=True)

y = x**2
x

tensor(3., requires_grad=True)

In [3]:
y

tensor(9., grad_fn=<PowBackward0>)

In [7]:
x.grad

tensor(6.)

In [9]:
import math

def dz_dx(x):
    return 2 * x * math.cos(x**2)

dz_dx(8)

6.2697156868728

In [10]:
x = torch.tensor(4.0,requires_grad=True)
y = x**2

z = torch.sin(y)

In [11]:
x

tensor(4., requires_grad=True)

In [12]:
y

tensor(16., grad_fn=<PowBackward0>)

In [13]:
z

tensor(-0.2879, grad_fn=<SinBackward0>)

In [16]:
z.backward()
print(x.grad)

tensor(-7.6613)


In [17]:
import torch

# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

In [18]:
# Manual we can check Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [19]:
loss

tensor(6.7012)

In [20]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [21]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


In [22]:
# uisng utograd

x = torch.tensor(6.7)
y = torch.tensor(0.0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

print(f"weight:{w}")
print(f"bias:{b}")

weight:1.0
bias:0.0


In [23]:
z = w*x+b
print(z)

tensor(6.7000, grad_fn=<AddBackward0>)


In [25]:
y_pred = torch.sigmoid(z)
print(y_pred)

# calculate loss
loss = binary_cross_entropy_loss(y_pred,y)
print(f"loss:{loss}")

tensor(0.9988, grad_fn=<SigmoidBackward0>)
loss:6.701176166534424


In [26]:
loss.backward()

print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


In [28]:
# clearing prevoius caluclated grad manually
x = torch.tensor(2.0,requires_grad=True)
print(x)

y = x**2
print(y)

y.backward()
x.grad

tensor(2., requires_grad=True)
tensor(4., grad_fn=<PowBackward0>)


tensor(4.)

In [30]:
x.grad.zero_()

tensor(0.)

In [32]:
# disabled gradient descent tracking during evaluation time
x.requires_grad_(False)
print(x)

y = x ** 2
print(y)

tensor(2.)
tensor(4.)


### ***Train Our Neural Network Pipeline (on real dataset)***

In [34]:
import numpy as np
import pandas as pd

import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [35]:
df.shape

(569, 33)

In [37]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
df.sample(6)

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
549,B,10.82,24.21,68.89,361.6,0.08192,0.06602,0.01548,0.00816,0.1976,...,13.030,31.45,83.90,505.6,0.12040,0.16330,0.06194,0.03264,0.3059,0.07626
386,B,12.21,14.09,78.78,462.0,0.08108,0.07823,0.06839,0.02534,0.1646,...,13.130,19.29,87.65,529.9,0.10260,0.24310,0.30760,0.09140,0.2677,0.08824
192,B,9.72,18.22,60.73,288.1,0.06950,0.02344,0.00000,0.00000,0.1653,...,9.968,20.83,62.25,303.8,0.07117,0.02729,0.00000,0.00000,0.1909,0.06559
534,B,10.96,17.62,70.79,365.6,0.09687,0.09752,0.05263,0.02788,0.1619,...,11.620,26.51,76.43,407.5,0.14280,0.25100,0.21230,0.09861,0.2289,0.08278
335,M,17.06,21.00,111.80,918.6,0.11190,0.10560,0.15080,0.09934,0.1727,...,20.990,33.15,143.20,1362.0,0.14490,0.20530,0.39200,0.18270,0.2623,0.07599
262,M,17.29,22.13,114.40,947.8,0.08999,0.12730,0.09697,0.07507,0.2108,...,20.390,27.24,137.90,1295.0,0.11340,0.28670,0.22980,0.15280,0.3067,0.07484


In [38]:
X = df.iloc[:,1:]
y = df.iloc[:,0] # unmaed column

In [39]:
# Train Test Split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.3,random_state=42
)

In [40]:
from numpy.random.mtrand import standard_cauchy
## Apply Scaling Factor
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [43]:
X_train

array([[-0.12348985, -0.29680142, -0.17050713, ..., -0.84082156,
        -0.8563616 , -0.76574773],
       [-0.22826757, -0.65795149, -0.25377521, ..., -0.37706655,
        -1.3415819 , -0.41480748],
       [ 0.14553402, -1.23056444,  0.24583328, ..., -0.04762652,
        -0.08997059,  0.4882635 ],
       ...,
       [ 0.03226081, -0.55578404, -0.08064356, ..., -1.26179013,
        -0.6828391 , -1.27672587],
       [-0.05552593,  0.10949242, -0.04684166, ...,  1.07924018,
         0.4755842 ,  1.25530227],
       [-0.56525537,  0.32333128, -0.619825  , ..., -0.61952313,
        -0.30366032, -0.84348042]])

In [46]:
# Aplly Label Encoding
encode = LabelEncoder()
y_train = encode.fit_transform(y_train)
y_test = encode.transform(y_test)

print(y_train)

[0 0 0 0 0 1 0 0 0 0 0 0 1 1 1 0 0 1 0 1 0 0 0 0 1 0 0 1 0 0 0 1 0 1 1 0 0
 0 1 0 0 0 0 1 0 0 0 0 0 1 0 1 1 0 0 1 0 0 0 0 0 0 0 1 1 1 0 0 1 0 0 1 0 1
 0 1 0 1 0 0 1 0 0 0 1 0 1 0 1 0 1 0 0 1 0 0 0 0 1 0 0 0 1 0 0 1 0 0 1 0 0
 0 0 0 0 0 1 0 0 0 1 0 1 0 0 0 1 0 1 1 0 0 1 0 1 1 1 0 0 0 1 0 0 1 0 1 0 0
 0 1 0 1 0 0 1 1 0 0 1 0 1 1 0 1 1 0 0 1 1 1 0 0 0 0 1 0 1 1 1 1 0 0 0 0 0
 0 0 0 1 1 0 0 1 0 0 0 0 0 1 0 0 1 1 0 1 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0
 0 1 0 0 1 0 1 1 1 0 1 0 0 1 1 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1 1 0 1 0 1 1
 0 0 1 0 1 1 0 1 1 0 0 1 0 1 0 0 1 0 0 1 1 1 0 0 0 1 1 0 1 1 0 0 0 1 0 1 1
 1 1 0 0 1 0 0 1 1 1 1 1 1 0 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 0 1 0 0 0 0 0 1
 1 1 0 0 1 0 0 1 1 1 1 0 0 1 1 0 0 0 1 1 1 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0
 0 1 0 0 0 0 0 0 1 0 0 1 0 0 1 1 1 0 1 1 0 1 0 0 0 0 1 0]


In [47]:
## Numpy Array ---> Pytorch Tensors

X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [48]:
X_train_tensor.shape

torch.Size([398, 30])

In [49]:
y_train_tensor.shape

torch.Size([398])

### Manual We Can Define Our Model & apply operations

In [52]:
# Build Our Simple Model
class Simple_NN():
  def __init__(self, X):

    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True) # weight
    self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True) # bias

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias # z = w*x+b
    y_pred = torch.sigmoid(z) # output
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss



In [53]:
# imporant params
learning_rate = 0.1
epochs = 25

# Training Pipeline
model = Simple_NN(X_train_tensor)

for epoch in range(epochs):
  y_pred = model.forward(X_train_tensor) # forward pass
  loss = model.loss_function(y_pred,y_train_tensor) # loss

  loss.backward() # backward pass

  # update our params
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 3.540318059861174
Epoch: 2, Loss: 3.411403799020852
Epoch: 3, Loss: 3.2784386865027657
Epoch: 4, Loss: 3.143739587473485
Epoch: 5, Loss: 3.005280894701768
Epoch: 6, Loss: 2.856840542025231
Epoch: 7, Loss: 2.7042113945969697
Epoch: 8, Loss: 2.5465680251880474
Epoch: 9, Loss: 2.3860473777067415
Epoch: 10, Loss: 2.222946684140729
Epoch: 11, Loss: 2.0591326868273225
Epoch: 12, Loss: 1.901347197956741
Epoch: 13, Loss: 1.7505330433299886
Epoch: 14, Loss: 1.6055521088195222
Epoch: 15, Loss: 1.4714509528384454
Epoch: 16, Loss: 1.3501288221502983
Epoch: 17, Loss: 1.239385580875928
Epoch: 18, Loss: 1.1391164801272609
Epoch: 19, Loss: 1.0536644614382098
Epoch: 20, Loss: 0.9854910195910722
Epoch: 21, Loss: 0.9328299331253602
Epoch: 22, Loss: 0.8931166035928191
Epoch: 23, Loss: 0.8634972153650057
Epoch: 24, Loss: 0.8413136355902657
Epoch: 25, Loss: 0.8243939486786058


In [54]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')


Accuracy: 0.6238842606544495


### ***Build Our Model (NN Module)***

In [67]:
import torch.nn as nn

class SimpleNN(nn.Module):
  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):

    out = self.linear(features)
    out = self.sigmoid(out)
    return out

In [69]:
# Training Pipeline
epochs = 30

# define loss function
loss_func = nn.BCELoss()

model = SimpleNN(X_train_tensor.shape[1])
optimizer = torch.optim.SGD(model.parameters(),lr=0.2)

# Convert X_train_tensor to float32 to match model's default dtype
X_train_tensor = X_train_tensor.float()

for epoch in range(epochs):
  # forward pass
  y_pred = model(X_train_tensor)

  # loss calculate
  loss = loss_func(y_pred, y_train_tensor.view(-1,1).float()) # Cast target to float as well

  # clear gradients
  optimizer.zero_grad()

  # backward pass
  loss.backward()

   # parameters update
  optimizer.step()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.881629228591919
Epoch: 2, Loss: 0.47947996854782104
Epoch: 3, Loss: 0.3743648827075958
Epoch: 4, Loss: 0.32110944390296936
Epoch: 5, Loss: 0.2872623801231384
Epoch: 6, Loss: 0.26320770382881165
Epoch: 7, Loss: 0.24493052065372467
Epoch: 8, Loss: 0.23041453957557678
Epoch: 9, Loss: 0.21851979196071625
Epoch: 10, Loss: 0.2085452377796173
Epoch: 11, Loss: 0.20003119111061096
Epoch: 12, Loss: 0.19266068935394287
Epoch: 13, Loss: 0.1862059384584427
Epoch: 14, Loss: 0.18049782514572144
Epoch: 15, Loss: 0.17540735006332397
Epoch: 16, Loss: 0.1708340048789978
Epoch: 17, Loss: 0.16669823229312897
Epoch: 18, Loss: 0.16293609142303467
Epoch: 19, Loss: 0.15949560701847076
Epoch: 20, Loss: 0.15633414685726166
Epoch: 21, Loss: 0.15341642498970032
Epoch: 22, Loss: 0.15071293711662292
Epoch: 23, Loss: 0.14819887280464172
Epoch: 24, Loss: 0.14585323631763458
Epoch: 25, Loss: 0.14365802705287933
Epoch: 26, Loss: 0.1415979117155075
Epoch: 27, Loss: 0.1396595537662506
Epoch: 28, Loss: 0.

In [71]:
# model evaluation
with torch.no_grad():
  # Convert X_test_tensor and y_test_tensor to float32 to match model's default dtype
  X_test_tensor = X_test_tensor.float()
  y_test_tensor = y_test_tensor.float()

  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5330870747566223


### ***Apply NN Module To Our Neural Network***

In [60]:
## Create Model Class
import torch
import torch.nn as nn

class Model(nn.Module):
  def __init__(self,input_features):

    super().__init__()
    # using contianers to make layer
    self.network = nn.Sequential(
        nn.Linear(input_features, 3),
        nn.ReLU(),
        nn.Linear(3, 1),
        nn.Sigmoid()
    )

  def forward(self,features):
      output = self.network(features)
      return output

In [61]:
# create dataset
features = torch.rand(10,5)

# create model
model = Model(features.shape[1])

# call model for forward pass
model(features)

tensor([[0.6415],
        [0.6668],
        [0.6642],
        [0.6293],
        [0.6522],
        [0.6353],
        [0.6836],
        [0.6529],
        [0.6149],
        [0.6569]], grad_fn=<SigmoidBackward0>)

In [63]:
!pip install torchinfo

In [64]:
from torchinfo import summary
summary(model, input_size=(10, 5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 1]                   4
│    └─Sigmoid: 2-4                      [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00